In [54]:
# IMPORT LIBRARIES

In [55]:
import pandas as pd
import sqlite3

In [56]:
# CONNECT DATABASE

In [57]:
conn = sqlite3.connect('ipl_data.db')
print("DataBase connected successfully")

DataBase connected successfully


In [58]:
# CATEGORIZE RCB MATCHES AS WIN/LOSS/NO RESULT

In [59]:
q1 =pd.read_sql_query( """
SELECT
    season, date, team1, team2, winner,
    CASE WHEN winner LIKE 'Royal Challengers%' THEN 'Win' 
         WHEN winner NOT LIKE 'Royal Challengers%' THEN 'Loss' 
         WHEN winner IS NULL THEN 'No Result'
         ELSE 'Tie'
    END as 'Outcome'
FROM matches
""",conn)
print("Match outcomes (first 10):")
display(q1.head(10))

Match outcomes (first 10):


,season,date,team1,team2,winner,Outcome
0,2007/08,2008-04-18,Royal Challengers Bengaluru,Kolkata Knight Riders,Kolkata Knight Riders,Loss
1,2007/08,2008-04-19,Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,Loss
2,2007/08,2008-04-19,Delhi Daredevils,Rajasthan Royals,Delhi Daredevils,Loss
3,2007/08,2008-04-20,Mumbai Indians,Royal Challengers Bengaluru,Royal Challengers Bangalore,Win
4,2007/08,2008-04-20,Kolkata Knight Riders,Deccan Chargers,Kolkata Knight Riders,Loss
5,2007/08,2008-04-21,Rajasthan Royals,Kings XI Punjab,Rajasthan Royals,Loss
6,2007/08,2008-04-22,Deccan Chargers,Delhi Daredevils,Delhi Daredevils,Loss
7,2007/08,2008-04-23,Chennai Super Kings,Mumbai Indians,Chennai Super Kings,Loss
8,2007/08,2008-04-24,Deccan Chargers,Rajasthan Royals,Rajasthan Royals,Loss
9,2007/08,2008-04-25,Kings XI Punjab,Mumbai Indians,Kings XI Punjab,Loss


In [60]:
# COUNT MATCHES BY OUTCOME

In [61]:
q1 = pd.read_sql_query("""
    SELECT
        CASE 
            WHEN winner LIKE 'Royal Challengers%' THEN 'Win' 
            WHEN winner IS NULL THEN 'No Result'
            WHEN winner NOT LIKE 'Royal Challengers%' THEN 'Loss' 
            ELSE 'Tie'
        END as Outcome,
        COUNT(*) as Match_Count
    FROM matches
    WHERE team1 = 'Royal Challengers Bengaluru' OR team2 = 'Royal Challengers Bengaluru'
    GROUP BY Outcome
    ORDER BY Match_Count DESC
""", conn)
print("Match outcomes count:")
display(q1)

Match outcomes count:


,Outcome,Match_Count
0,Loss,129
1,Win,123
2,No Result,3


In [62]:
# TOSS ANALYSIS — DID RCB WIN THE TOSS?

In [63]:
q1 =pd.read_sql_query( """
SELECT
    season, toss_winner,
    CASE WHEN toss_winner like 'Royal Challengers%' THEN 'YES' ELSE 'NO'
    END as 'rcb_won_toss',
    winner,
    CASE WHEN winner LIKE 'Royal Challengers%' THEN 'Win' 
         WHEN winner IS NULL THEN 'No Result'
         WHEN winner NOT LIKE 'Royal Challengers%' THEN 'Loss' 
         ELSE 'Tie'
    END as 'Outcome'
FROM matches
""",conn)
print("Toss analysis (first 10):")
display(q1.head())

Toss analysis (first 10):


,season,toss_winner,rcb_won_toss,winner,Outcome
0,2007/08,Royal Challengers Bangalore,YES,Kolkata Knight Riders,Loss
1,2007/08,Chennai Super Kings,NO,Chennai Super Kings,Loss
2,2007/08,Rajasthan Royals,NO,Delhi Daredevils,Loss
3,2007/08,Mumbai Indians,NO,Royal Challengers Bangalore,Win
4,2007/08,Deccan Chargers,NO,Kolkata Knight Riders,Loss


In [64]:
# WIN PERCENTAGE BY TOSS OUTCOME

In [65]:
q1 =pd.read_sql_query( """
SELECT 
    CASE 
        WHEN toss_winner like 'Royal Challengers%' THEN 'Won Toss'
        ELSE 'Lost Toss'
    END AS toss_result,
    COUNT(*) AS total_matches,
    COUNT(CASE WHEN winner like 'Royal Challengers%' THEN 1 END) AS wins,
    ROUND(COUNT(CASE WHEN winner like 'Royal Challengers%' THEN 1 END) * 100.0 / COUNT(*), 1) AS win_pct
FROM matches
WHERE team1 like 'Royal Challengers%'
   OR team2 like 'Royal Challengers%'
GROUP BY toss_result
""",conn)
display(q1.head())

,toss_result,total_matches,wins,win_pct
0,Lost Toss,134,62,46.3
1,Won Toss,121,61,50.4


In [66]:
# BATTING FIRST vs CHASING WINS

In [67]:
q = pd.read_sql_query("""
SELECT 
    CASE 
        WHEN result = 'runs' THEN 'Batting First Win'
        WHEN result = 'wickets' THEN 'Chasing Win'
        ELSE 'Other'
    END AS win_type,
    COUNT(*) AS total_wins
FROM matches
WHERE winner like 'Royal Challengers%'
GROUP BY win_type
ORDER BY total_wins DESC
""", conn)
display(q)

,win_type,total_wins
0,Chasing Win,64
1,Batting First Win,57
2,Other,2


In [68]:
# CLOSE CONNECTION

In [69]:
conn.close()
print("Connection closed.")

Connection closed.
